# 4. Analytical scores and histogram coarsening


    Compute the analytical score at each tested signal strength, transform it to
    a bounded scalar, and compare increasingly fine Poisson histograms with the
    unbinned likelihood. The construction parameter is called \(\eta\); the
    fitted physical parameter remains \(\mu\). Run notebooks 1–3 first.

### Runtime and persistent run

In Colab, choose **Runtime → Change runtime type → GPU** for notebooks 2–3.
PyTorch uses the GPU for training; the JAX likelihood fits run on the CPU.
Setup removes preinstalled JAX CUDA plugins that are incompatible with the pinned
JAX version, then checks the JAX runtime before training. If an earlier session
reported a JAX plugin error, restart the runtime once before rerunning setup.
Keep the same run name: saved samples and trained networks are reused.
The source checkout lives in the temporary runtime; **datasets, checkpoints,
workspace files, and results live in Google Drive**. Use the same `RUN_NAME`
and `MODE` in every notebook. Set `MODE = "smoke"` for a short workflow check;
the production default generates five million events **per sample**.
Use `CONFIG_OVERRIDES` to change a computational budget without editing source,
for example `{"quadrature_per_process": 500_000, "epochs": 150}`. Use the same
overrides in all five notebooks, and choose a new `RUN_NAME` when changing them.

This repository may be private. Grant Colab access when opening it from GitHub.
For the runtime download, either save a GitHub fine-grained token with **Contents:
read** access to this repository as the Colab secret `GITHUB_TOKEN`, or enter it
in the hidden prompt. The token is used only in a Python HTTP header and is not
written into files, Git remotes, shell arguments, or notebook output. Alternatively,
upload a repository ZIP to `/content/poodemo_source.zip`; no token is then needed.
An existing `/content/poodemo` checkout is reused. Delete that checkout to fetch
new source after updating the repository; your Drive run is separate.
For this NI-only revision, use the new default `RUN_NAME = "demo-ni-v1"`.
If you ran an earlier version, start a fresh Colab runtime (or refresh the
temporary source checkout and restart the runtime) before proceeding. Existing
run manifests from the earlier model are incompatible and are not reused.

In [ ]:
import os
import sys
from pathlib import Path

RUN_NAME = "demo-ni-v1"
MODE = os.environ.get("POODEMO_MODE", "production")  # production or smoke
CONFIG_OVERRIDES = {}  # e.g. {"quadrature_per_process": 500_000, "epochs": 150}

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # JAX likelihood fits use the validated CPU runtime. This does not change
    # PyTorch's CUDA visibility or GPU training.
    os.environ["JAX_PLATFORMS"] = "cpu"
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = Path(os.environ.get("POODEMO_ROOT", f"/content/drive/MyDrive/poodemo/runs/{RUN_NAME}"))
    REPO_DIR = Path("/content/poodemo")
    if not (REPO_DIR / "pyproject.toml").exists():
        import getpass
        import io
        import shutil
        import tarfile
        import tempfile
        import urllib.error
        import urllib.request
        import zipfile

        uploaded_zip = Path("/content/poodemo_source.zip")
        if uploaded_zip.exists():
            archive_bytes = uploaded_zip.read_bytes()
            archive_kind = "zip"
        else:
            archive_url = "https://api.github.com/repos/rafaellopesdesa/poodemo/tarball/main"

            def fetch_source(token=None):
                headers = {"Accept": "application/vnd.github+json", "User-Agent": "poodemo-colab"}
                if token:
                    headers["Authorization"] = "Bearer " + token
                request = urllib.request.Request(archive_url, headers=headers)
                with urllib.request.urlopen(request, timeout=180) as response:
                    return response.read()

            try:
                archive_bytes = fetch_source()
            except urllib.error.HTTPError as error:
                if error.code not in (401, 403, 404):
                    raise RuntimeError(f"Repository download failed (HTTP {error.code}).") from None
                token = None
                try:
                    from google.colab import userdata
                    token = userdata.get("GITHUB_TOKEN")
                except Exception:
                    pass
                if not token:
                    token = getpass.getpass("GitHub token with read access to poodemo: ").strip()
                if not token:
                    raise RuntimeError("Upload /content/poodemo_source.zip or provide read access.")
                try:
                    archive_bytes = fetch_source(token)
                except urllib.error.HTTPError as auth_error:
                    raise RuntimeError(f"Repository download failed (HTTP {auth_error.code}); check token access.") from None
                finally:
                    token = None
            archive_kind = "tar"

        # Validate archive paths and reject links before extracting the source.
        with tempfile.TemporaryDirectory(prefix="poodemo-source-") as temp:
            staging = Path(temp).resolve()

            def safe_destination(name):
                destination = (staging / name).resolve()
                if not destination.is_relative_to(staging):
                    raise RuntimeError("Unsafe path in source archive.")
                return destination

            if archive_kind == "zip":
                with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
                    for entry in archive.infolist():
                        safe_destination(entry.filename)
                        if (entry.external_attr >> 16) & 0o170000 == 0o120000:
                            raise RuntimeError("Symbolic links are not supported in the source ZIP.")
                    archive.extractall(staging)
            else:
                with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:gz") as archive:
                    for entry in archive.getmembers():
                        safe_destination(entry.name)
                        if not (entry.isfile() or entry.isdir()):
                            raise RuntimeError("Links and special files are not supported in the source archive.")
                    archive.extractall(staging)
            candidates = [p.parent for p in staging.rglob("pyproject.toml") if (p.parent / "poodemo").is_dir()]
            if len(candidates) != 1:
                raise RuntimeError("The uploaded archive must contain one poodemo repository checkout.")
            shutil.copytree(candidates[0], REPO_DIR, dirs_exist_ok=True)
        del archive_bytes

    import subprocess
    if os.environ.get("POODEMO_SKIP_INSTALL") != "1":
        # JAX discovers plugins even with JAX_PLATFORMS=cpu. Colab can retain
        # newer CUDA plugins when pip installs an older jax/jaxlib pair.
        subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                               "jax-cuda12-plugin", "jax-cuda12-pjrt",
                               "jax-cuda13-plugin", "jax-cuda13-pjrt"])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements-colab.txt")])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])

        # Check the live notebook kernel before any expensive network training.
        import jax
        import jaxlib
        if jax.__version__ != "0.5.3" or jaxlib.__version__ != "0.5.3":
            raise RuntimeError("JAX packages are stale in this kernel. Restart the runtime and rerun setup.")
        jax.config.update("jax_platforms", "cpu")
        jax.config.update("jax_enable_x64", True)
        jax_devices = jax.devices()
        if any(device.platform != "cpu" for device in jax_devices):
            raise RuntimeError("Expected the CPU JAX backend for likelihood fits. Restart the runtime and rerun setup.")
        print(f"JAX {jax.__version__} / jaxlib {jaxlib.__version__}: {jax_devices}")

    # A new editable install is not activated in an already-running kernel.
    # Expose the inner package directly, including after a failed import cached
    # the outer /content/poodemo checkout as a namespace package.
    import importlib
    source_root = str(REPO_DIR.resolve())
    if source_root in sys.path:
        sys.path.remove(source_root)
    sys.path.insert(0, source_root)
    importlib.invalidate_caches()
    cached_package = sys.modules.get("poodemo")
    cached_file = getattr(cached_package, "__file__", None)
    expected_init = REPO_DIR / "poodemo" / "__init__.py"
    if cached_package is not None and (cached_file is None or Path(cached_file).resolve() != expected_init.resolve()):
        for module_name in list(sys.modules):
            if module_name == "poodemo" or module_name.startswith("poodemo."):
                del sys.modules[module_name]
else:
    # Locally: pip install -r requirements-colab.txt && pip install -e .
    ROOT = Path(os.environ.get("POODEMO_ROOT", str(Path.home() / "poodemo-runs" / RUN_NAME)))
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "poodemo" / "pipeline.py").exists():
            sys.path.insert(0, str(candidate))
            break

ROOT.mkdir(parents=True, exist_ok=True)
print(f"Run directory: {ROOT}")
print(f"Mode: {MODE}")

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from poodemo.pipeline import create_run

run = create_run(ROOT, mode=MODE, overrides=CONFIG_OVERRIDES)
plt.rcParams.update({"figure.figsize": (7, 4.5), "axes.grid": True, "grid.alpha": 0.2})
display(pd.Series(run.config, name="configuration"))

### The score, including the yield factors

In the selected phase space define normalized process densities \(p_j\)
and their selected yields \(\lambda_j\). At \(\alpha_{NI}=0\),
\[
D_\mu(x)=(\mu-\sqrt\mu)\lambda_Sp_S(x)
+\sqrt\mu\lambda_{SBI}p_{SBI}(x)
+(1-\sqrt\mu)\lambda_Bp_B(x)+\lambda_{NI}p_{NI}(x),
\]
\[
\Lambda_\mu=(\mu-\sqrt\mu)\lambda_S
+\sqrt\mu\lambda_{SBI}+(1-\sqrt\mu)\lambda_B+\lambda_{NI},
\qquad p(x;\mu)=D_\mu(x)/\Lambda_\mu.
\]
In particular, **there is no density factor in \(\Lambda_\mu\)**.
For \(\eta>0\),
\[
D'_\eta(x)=\frac{(2\sqrt\eta-1)\lambda_Sp_S(x)
       +\lambda_{SBI}p_{SBI}(x)-\lambda_Bp_B(x)}{2\sqrt\eta},
\]
\[
\boxed{o_\eta(x)=\left.\partial_\mu\log p(x;\mu)\right|_\eta
         =\frac{D'_\eta(x)}{D_\eta(x)}
          -\frac{\Lambda'_\eta}{\Lambda_\eta}.}
\]
The constant normalization term ensures zero mean under the selected
normalized distribution. Keeping the event count separately preserves
rate information. The bounded observable is
\(z_\eta=1/[1+\exp(-o_\eta/a)]\), with **\(a=1\)**.

### Freeze the observable inside each likelihood fit

For the test of \(\mu_0\), construct \(z_{\eta=\mu_0}\) at nominal
\(\alpha_{NI}=0\), then keep its bin edges, event assignments, and Asimov data
fixed while fitting all physical alternatives \(\mu\). Thus,
\[
t(\mu_0)=-2\log\frac{L(\mu_0;\eta=\mu_0)}
                           {\max_\mu L(\mu;\eta=\mu_0)}.
\]
The Asimov data must also be re-histogrammed when \(\eta\) changes.
The binned likelihood is an independent Poisson likelihood for disjoint
bins of a Poisson point process. This notebook fixes \(\alpha_{NI}=0\),
where the scalar score has its clean local guarantee.

In [ ]:
from poodemo.pipeline import run_binning_study

binning = run_binning_study(run)
display(binning['scans'])
display(binning['fidelity'])

In [ ]:
def plot_scans(frame, title, filename, group_columns=None):
    """Plot saved scan tables; adapt grouping to the columns present."""
    x_column = next((c for c in ["mu", "mu_test", "mu0"] if c in frame.columns), None)
    y_column = next((c for c in ["q", "t_mu", "test_statistic", "ts"] if c in frame.columns), None)
    if x_column is None or y_column is None:
        raise ValueError(f"Expected mu and q scan columns; found {list(frame.columns)}")
    if group_columns is None:
        group_columns = [c for c in ["method", "model", "systematics", "nuisances", "n_bins", "bins", "observable"] if c in frame.columns]
    groups = frame.groupby(group_columns, dropna=False, sort=False) if group_columns else [("scan", frame)]
    fig, ax = plt.subplots()
    for label, group in groups:
        group = group.sort_values(x_column)
        values = label if isinstance(label, tuple) else (label,)
        parts = []
        for column, value in zip(group_columns, values):
            if pd.isna(value):
                continue
            if column == "systematics":
                parts.append("profiled" if value else "stat only")
            elif column in ("n_bins", "bins"):
                parts.append(f"{int(value)} bins")
            else:
                parts.append(str(value))
        ax.plot(group[x_column], group[y_column], marker=".", ms=3, label=" | ".join(parts))
    ax.axhline(1.0, color="0.45", ls=":", lw=1)
    ax.set(xlabel=r"Tested signal strength $\mu_0$", ylabel=r"$t(\mu_0)$", title=title)
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=8)
    fig.tight_layout()
    output = ROOT / "plots"
    output.mkdir(exist_ok=True)
    fig.savefig(output / filename, bbox_inches="tight")
    plt.show()
    return fig

In [ ]:
plot_scans(binning['scans'], 'Score histograms: progressively finer binning', '04_binning_scans.pdf');

### What convergence can and cannot show

Two information losses must be distinguished:

1. **Compression** from \(x\) to \(o_\eta\). Before binning, the exact
   score retains the local Fisher information at \(\mu=\eta\).
2. **Finite binning** of that score. At the anchor its shape-information
   loss is \(E_\eta[\operatorname{Var}_\eta(o_\eta\mid\text{bin})]\).

Finer nested bins reduce the second loss. They do not automatically
restore global likelihood information removed by scalar compression.
Consequently, fine-bin curves should approach the **unbinned score
experiment**, and can approximate the full unbinned scan very well
locally; exact equality over a broad interference scan is not guaranteed.
The tables report deviations rather than assuming convergence to zero.

In a fixed generating Asimov experiment, \(\mu_*\) and the tested
\(\eta\) differ away from the minimum. This is precisely where local
optimality must not be mistaken for global sufficiency.

In [ ]:
fidelity = binning["fidelity"]
display(fidelity)
eta_near_truth = min(fidelity["eta"].unique(), key=lambda e: abs(e - run.config["asimov_mu"]))
at_anchor = fidelity[np.isclose(fidelity["eta"], eta_near_truth)].sort_values("n_bins")
y_columns = [c for c in fidelity if "information_fraction" in c]
if y_columns:
    ax = at_anchor.plot(x="n_bins", y=y_columns, marker="o", logx=True)
    ax.set(xlabel="Number of bins", ylabel="Retained Fisher information fraction",
           title=fr"Information retained at $\eta={eta_near_truth:.3g}$")
    ax.figure.tight_layout()
    (ROOT / "plots").mkdir(exist_ok=True)
    ax.figure.savefig(ROOT / "plots" / "04_binning_diagnostics.pdf", bbox_inches="tight")
    plt.show()

Next: **05_spline_templates.ipynb**.